## Imports

In [5]:
import json
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator, ConfigDict
import re

## Defining the Fashion Schema (Guardrails)

In [6]:
# 1 color sub model
class ColorDetail(BaseModel): 
    hex: str = Field(..., description='Hex color code')
    name: str

    @field_validator('hex')
    @classmethod
    def validate_hex(cls, v: str) -> str:
        # 2026 standard hex validation: ensures '#' prefix and 6 characters
        v = v.strip() 
        if not v.startswith('#'):
            v = f'#{v}'
        if not re.match(r'^#[0-9A-Fa-f]{6}$', v):
            raise ValueError(f'Invalid Hex Format: {v}')
        return v

# 2 recommendation sub model
class OutfitItem(BaseModel): 
    color_name: str
    hex: str
    reason: str

# 3 main data model (our ai core)
class StyleStepAnalysis(BaseModel): 
    model_config = ConfigDict(extra='ignore')   # ignore extra fields if Gemma 4 model hallucinated

    model_style: str
    material_composition: str
    perceived_vibe: str

    # nested objects
    dominant_color: ColorDetail
    accent_colors: List[ColorDetail]
    suggested_trousers: List[OutfitItem]
    suggested_shirts: List[OutfitItem]

    # metadata
    lighting_assessment: str
    confidence_score: float = Field(ge=0, le=1)

## loading research results

In [7]:
from pathlib import Path

In [8]:
results_path = f'{Path().resolve().parent}/data/results.json'

try: 
    with open(results_path, 'r') as f:
        raw_data = json.load(f) 
    print(f'Successfully loaded {len(raw_data)} test items.')
except FileNotFoundError:
    print('Error: FIle not found in the directory!!') 

Successfully loaded 5 test items.


## Validation Engine

In [11]:
def validate_research_data(data_list): 
    validated_results = []
    failed_items = [] 

    print('-'* 30)

    for i, item in enumerate(data_list): 
        try: 
            # gemma 4 e2b returns nested keys: 'shoe_analysis', 'color_pallete', etc.
            # we flatten/ map them into our StyleStepAnalysis model here

            # extract data from the nested structure
            processed_data = {
                'model_style': item['shoe_analysis']['model_style'],
                'material_composition': item['shoe_analysis']['material_composition'],
                'perceived_vibe': item['shoe_analysis']['perceived_vibe'],
                "dominant_color": item['color_palette']['dominant'],
                "accent_colors": item['color_palette']['accents'],
                "suggested_trousers": item['coordinated_outfits']['trousers'],
                "suggested_shirts": item['coordinated_outfits']['shirts'],
                "lighting_assessment": item['metadata']['lighting_assessment'],
                "confidence_score": item['metadata']['confidence_score']
            }

            valid_outfit = StyleStepAnalysis(**processed_data)
            validated_results.append(valid_outfit)
            print(f'✅ ITEM {i}: VALID ({valid_outfit.model_style})')
        
        except Exception as e: 
            print(f"❌ ITEM {i}: FAILED | Reason: {str(e)[:100]}...")
            failed_items.append({"index": i, "error": str(e)})

    print("-" * 30)
    print(f"Summary: {len(validated_results)} Passed, {len(failed_items)} Failed.")
    return validated_results

# Execute the parser
validated_data = validate_research_data(raw_data)            


------------------------------
✅ ITEM 0: VALID (knit_sneaker)
✅ ITEM 1: VALID (low_top_sneaker)
✅ ITEM 2: VALID (casual_sneaker)
✅ ITEM 3: VALID (stiletto_pump)
✅ ITEM 4: VALID (retro_sneaker)
------------------------------
Summary: 5 Passed, 0 Failed.


## inspecting clean data

In [16]:
if validated_data:
    first_match = validated_data[2]

    print(f'SHOE: {first_match.model_style.upper()}')
    print(f'PRIMARY COLOR: {first_match.dominant_color.name} ({first_match.dominant_color.hex})')  
    print(f'BEST TROUSER: {first_match.suggested_trousers[0].color_name}') 
    print(f'WHY: {first_match.suggested_trousers[0].reason}') 

SHOE: CASUAL_SNEAKER
PRIMARY COLOR: Off-White Cream (#E0E0D8)
BEST TROUSER: Stone Beige
WHY: Neutral base for high contrast with olive accents, aligning with 2026 muted earth tones.
